# 06 — Customer Segmentation Analysis
**Hotel Booking Demand Dataset**

This notebook segments hotel guests into meaningful business categories and
compares key performance indicators (KPIs) across segments to surface
actionable revenue management and marketing insights.

**Segmentation dimensions:**
1. **Business type:** Corporate vs Leisure (by `market_segment`)
2. **Booking timing:** Early Planners vs Last-Minute Bookers (by `lead_time`)

**KPIs per segment:** Average ADR, Cancellation Rate, Average Stay Length, Average Revenue per Booking


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

CORP_COLOR    = "#1f77b4"
LEISURE_COLOR = "#ff7f0e"
EARLY_COLOR   = "#9467bd"
LM_COLOR      = "#2ca02c"


In [ ]:
df = pd.read_csv("../data/cleaned/hotel_cleaned.csv",
                 parse_dates=["reservation_status_date", "arrival_date"])

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print("market_segment distribution:")
print(df["market_segment"].value_counts().to_string())


---
## 1 — Segment 1: Corporate vs Leisure

In [ ]:
# Map market segments to Corporate or Leisure
CORPORATE_SEGS = {"Corporate", "Aviation"}

df["biz_type"] = df["market_segment"].apply(
    lambda x: "Corporate" if x in CORPORATE_SEGS else "Leisure"
)

print("Segment counts:")
counts = df["biz_type"].value_counts()
for seg, n in counts.items():
    print(f"  {seg:<12}: {n:>6,}  ({n/len(df)*100:.1f}%)")

print()
print("Mapping used:")
for seg in df["market_segment"].unique():
    biz = "Corporate" if seg in CORPORATE_SEGS else "Leisure"
    print(f"  {seg:<22} -> {biz}")


---
## 2 — Segment 2: Early Planners vs Last-Minute Bookers

In [ ]:
def classify_timing(lead_time):
    if lead_time <= 30:   return "Last-Minute"
    elif lead_time > 90:  return "Early Planner"
    else:                 return "Mid-Range"

df["timing_seg"] = df["lead_time"].apply(classify_timing)

print("Timing segment counts:")
timing_counts = df["timing_seg"].value_counts()
for seg, n in timing_counts.items():
    print(f"  {seg:<15}: {n:>6,}  ({n/len(df)*100:.1f}%)")

print()
print("Lead time thresholds:")
print("  Last-Minute   : lead_time  <= 30 days")
print("  Mid-Range     : 31 <= lead_time <= 90 days")
print("  Early Planner : lead_time  >  90 days")


---
## 3 — KPIs per Segment

In [ ]:
def segment_kpis(data, group_col):
    return (
        data.groupby(group_col)
            .agg(
                Bookings          = ("adr",                "count"),
                Avg_ADR           = ("adr",                "mean"),
                Cancel_Rate_pct   = ("is_canceled",        "mean"),
                Avg_Nights        = ("total_nights",       "mean"),
                Avg_Revenue       = ("revenue_per_booking","mean"),
                Total_Revenue     = ("revenue_per_booking","sum"),
            )
            .assign(
                Cancel_Rate_pct = lambda x: (x["Cancel_Rate_pct"] * 100).round(2),
                Avg_ADR         = lambda x: x["Avg_ADR"].round(2),
                Avg_Nights      = lambda x: x["Avg_Nights"].round(2),
                Avg_Revenue     = lambda x: x["Avg_Revenue"].round(2),
                Total_Revenue   = lambda x: x["Total_Revenue"].round(0).astype(int),
                Share_pct       = lambda x: (x["Bookings"] / x["Bookings"].sum() * 100).round(1),
            )
    )

biz_kpis    = segment_kpis(df, "biz_type")
timing_kpis = segment_kpis(df[df["timing_seg"].isin(["Early Planner","Last-Minute"])],
                             "timing_seg")

print("=== Corporate vs Leisure ===")
print(biz_kpis.to_string())
print()
print("=== Early Planner vs Last-Minute ===")
print(timing_kpis.to_string())


### Styled Comparison Tables

In [ ]:
def make_display_table(kpis_df):
    display = kpis_df.copy()
    display["Avg_ADR"]       = display["Avg_ADR"].apply(lambda x: f"£{x:.2f}")
    display["Cancel_Rate_pct"] = display["Cancel_Rate_pct"].apply(lambda x: f"{x:.2f}%")
    display["Avg_Nights"]    = display["Avg_Nights"].apply(lambda x: f"{x:.2f}")
    display["Avg_Revenue"]   = display["Avg_Revenue"].apply(lambda x: f"£{x:.2f}")
    display["Total_Revenue"] = display["Total_Revenue"].apply(lambda x: f"£{x:,}")
    display["Bookings"]      = display["Bookings"].apply(lambda x: f"{x:,}")
    display["Share_pct"]     = display["Share_pct"].apply(lambda x: f"{x:.1f}%")
    display.columns = ["Bookings","Avg ADR","Cancel Rate","Avg Nights",
                        "Avg Revenue","Total Revenue","Share"]
    return display

print("Corporate vs Leisure — KPI Summary:")
print()
print(make_display_table(biz_kpis).to_string())
print()
print("Early Planner vs Last-Minute — KPI Summary:")
print()
print(make_display_table(timing_kpis).to_string())


---
## 4 — Grouped Bar Charts: KPI Comparison

In [ ]:
METRICS = [
    ("Avg_ADR",         "Average ADR (£)",            "£{:.0f}"),
    ("Cancel_Rate_pct", "Cancellation Rate (%)",       "{:.1f}%"),
    ("Avg_Nights",      "Avg Stay Length (nights)",    "{:.2f}"),
    ("Avg_Revenue",     "Avg Revenue per Booking (£)", "£{:.0f}"),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("Customer Segment KPI Comparison",
             fontsize=14, fontweight="bold", y=1.02)

# ── Row 1: Corporate vs Leisure ──────────────────────────────────────────────
for col_idx, (metric, title, fmt) in enumerate(METRICS):
    ax = axes[0, col_idx]
    segs   = biz_kpis.index.tolist()
    values = biz_kpis[metric].values
    colors = [CORP_COLOR, LEISURE_COLOR]

    bars = ax.bar(segs, values, color=colors, edgecolor="white", width=0.5)
    ax.set_title(f"Corp vs Leisure\n{title}", fontsize=10, fontweight="bold")
    ax.set_ylabel(title, fontsize=9)

    if "£" in fmt:
        ax.yaxis.set_major_formatter(
            mtick.FuncFormatter(lambda v, _: f"£{v:.0f}"))
    elif "%" in fmt:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())

    ax.set_ylim(0, max(values) * 1.25)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(values)*0.02,
                fmt.format(val),
                ha="center", va="bottom", fontsize=10, fontweight="bold")

# ── Row 2: Early Planner vs Last-Minute ─────────────────────────────────────
for col_idx, (metric, title, fmt) in enumerate(METRICS):
    ax = axes[1, col_idx]
    segs   = timing_kpis.index.tolist()
    values = timing_kpis[metric].values
    colors = [EARLY_COLOR if s == "Early Planner" else LM_COLOR for s in segs]

    bars = ax.bar(segs, values, color=colors, edgecolor="white", width=0.5)
    ax.set_title(f"Early vs Last-Minute\n{title}", fontsize=10, fontweight="bold")
    ax.set_ylabel(title, fontsize=9)

    if "£" in fmt:
        ax.yaxis.set_major_formatter(
            mtick.FuncFormatter(lambda v, _: f"£{v:.0f}"))
    elif "%" in fmt:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())

    ax.set_ylim(0, max(values) * 1.25)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(values)*0.02,
                fmt.format(val),
                ha="center", va="bottom", fontsize=10, fontweight="bold")

# Row labels
axes[0, 0].set_ylabel("Corp vs Leisure\nAvg ADR (£)", fontsize=9)
axes[1, 0].set_ylabel("Early vs Last-Minute\nAvg ADR (£)", fontsize=9)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=CORP_COLOR,    label="Corporate"),
    Patch(facecolor=LEISURE_COLOR, label="Leisure"),
    Patch(facecolor=EARLY_COLOR,   label="Early Planner"),
    Patch(facecolor=LM_COLOR,      label="Last-Minute"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=4,
           bbox_to_anchor=(0.5, -0.03), fontsize=10, frameon=True)

plt.tight_layout()
plt.savefig("../visuals/segment_kpi_comparison.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved -> visuals/segment_kpi_comparison.png")


In [ ]:
# Normalise KPIs to 0-1 for a side-by-side profile comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Segment Profiles — Normalised KPIs (0=lowest, 1=highest)",
             fontsize=13, fontweight="bold", y=1.02)

norm_metrics = ["Avg_ADR", "Cancel_Rate_pct", "Avg_Nights", "Avg_Revenue"]
metric_labels = ["Avg ADR", "Cancel Rate", "Avg Nights", "Avg Revenue"]

for ax, kpis, seg_colors, title in [
    (axes[0], biz_kpis,    [CORP_COLOR, LEISURE_COLOR], "Corporate vs Leisure"),
    (axes[1], timing_kpis, [EARLY_COLOR, LM_COLOR],     "Early Planner vs Last-Minute"),
]:
    norm = kpis[norm_metrics].copy()
    for col in norm_metrics:
        col_min, col_max = norm[col].min(), norm[col].max()
        norm[col] = (norm[col] - col_min) / (col_max - col_min + 1e-9)

    x = np.arange(len(metric_labels))
    width = 0.35
    for i, (seg, color) in enumerate(zip(norm.index, seg_colors)):
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, norm.loc[seg, norm_metrics].values,
                      width=width, color=color, edgecolor="white",
                      alpha=0.85, label=seg)

    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=10)
    ax.set_ylabel("Normalised Score (0–1)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.25)
    ax.legend(fontsize=10)

    # Raw value annotations
    for i, (seg, color) in enumerate(zip(kpis.index, seg_colors)):
        for j, col in enumerate(norm_metrics):
            raw_val = kpis.loc[seg, col]
            fmt = "£{:.0f}" if col in ["Avg_ADR","Avg_Revenue"] else (
                  "{:.1f}%" if "pct" in col else "{:.1f}")
            offset = (i - 0.5) * width
            ax.text(j + offset, norm.loc[seg, col] + 0.04,
                    fmt.format(raw_val),
                    ha="center", va="bottom", fontsize=8, fontweight="bold",
                    color=color)

plt.tight_layout()
plt.savefig("../visuals/segment_profiles_normalised.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved -> visuals/segment_profiles_normalised.png")


---
## 5 — Key Findings: Customer Segment Insights

### 5.1 Corporate vs Leisure

| KPI | Corporate | Leisure | Delta |
|-----|-----------|---------|-------|
| Bookings | 4,354 (5.1%) | 81,244 (94.9%) | — |
| **Avg ADR** | **£71.01** | **£109.46** | Leisure +54% |
| **Cancel Rate** | **13.00%** | **29.00%** | Leisure +16 pp |
| **Avg Nights** | **2.12** | **3.75** | Leisure +77% |
| **Avg Revenue** | **£155.34** | **£411.31** | Leisure +165% |

**Corporate guests pay significantly less but are dramatically more reliable:**
- Corporate bookings cancel at only **13%** vs **29%** for leisure — less than half the rate.
- Despite lower ADR, corporate bookings represent predictable, committed revenue.
- Leisure bookings drive 94.9% of volume and 165% more revenue per booking — but 
  also carry most of the cancellation risk.

**Strategic implication:** Corporate segment is a natural hedge against peak-season
cancellations. Hotels should cultivate corporate accounts to build a stable revenue
floor, while applying cancellation mitigation strategies primarily to the leisure
segment where risk is concentrated.

---

### 5.2 Early Planners vs Last-Minute Bookers

| KPI | Early Planner (>90d) | Last-Minute (≤30d) | Delta |
|-----|---------------------|-------------------|-------|
| Bookings | 29,762 (34.7%) | 33,319 (38.9%) | — |
| **Avg ADR** | **£109.57** | **£103.35** | Early +6% |
| **Cancel Rate** | **37.00%** | **17.00%** | Early +20 pp |
| **Avg Nights** | **4.77** | **2.60** | Early +83% |
| **Avg Revenue** | **£518.96** | **£275.90** | Early +88% |

**Early planners generate almost double the revenue per booking — but cancel at more
than twice the rate of last-minute bookers:**
- Early planners book **4.77 nights** on average vs **2.60** for last-minute — longer
  holidays committed far in advance.
- However, at **37% cancellation rate**, more than 1 in 3 early planner bookings 
  never materialises as actual revenue.
- Last-minute bookers cancel at only **17%** — they've already decided to travel 
  and are booking to confirm logistics.

**Strategic implication:** Early Planner bookings are high-value but high-risk. The
optimal strategy is a dual approach:
1. For Early Planners: incentivise non-refundable rates or partial deposits to 
   convert uncertain future intent into committed revenue.
2. For Last-Minute: use dynamic pricing to capture premium on their lower price 
   sensitivity (they *need* to book now).

---

### 5.3 Revenue Manager's Segment Matrix

| Segment | Revenue Potential | Cancellation Risk | Priority Action |
|---------|-----------------|-------------------|----------------|
| **Corporate** | 🟡 Moderate (£155/booking) | 🟢 Low (13%) | Grow this segment; offer negotiated rates |
| **Leisure** | 🟢 High (£411/booking) | 🔴 High (29%) | Mitigation strategies; non-refundable offers |
| **Early Planner** | 🟢 Very High (£519/booking) | 🔴 Very High (37%) | Deposit requirements; re-confirmation campaigns |
| **Last-Minute** | 🟡 Moderate (£276/booking) | 🟢 Low (17%) | Premium last-minute pricing; upsell add-ons |
